<a href="https://colab.research.google.com/github/boss-defender/Born-Baby-Ai/blob/main/Just_born_Baby_Ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 BabyFlash: DeepSeek V4.1 Flash Multi-Modal Foundation Model
### Automated Google Drive Smart Checkpoint & Auto-Resume Engine
*1-Click Training for Google Colab (Free T4 GPU or CPU) with Zero Lost Progress across Disconnects.*


In [ ]:
# @title ⚙️ Cell 1: Dependencies, Automated Drive Mounting & UI Form { run: "auto" }
# @markdown Configure your training dataset, task type, and model scale using the visual form below.

# @markdown ### 🧠 Model Identifier & Capacity
MODEL_NAME = "Baby-Flash"  # @param ["Baby-Tiny", "Baby-Flash", "My-Custom-BabyAI"]
TASK_TYPE = "text"  # @param ["text", "vision", "reasoning", "coding", "tool_use"]

# @markdown ### 📂 Dataset Source & Location
DATASET_SOURCE = "Hugging Face"  # @param ["Hugging Face", "Custom Upload / Drive"]
DATASET_PATH = "agentlans/li2017dailydialog"  # @param {type:"string"}
MAX_SAMPLES = None  # @param {type:"integer"}
MAX_SEQ_LENGTH = 128  # @param {type:"integer"}

# @markdown ### 💾 Google Drive Smart Checkpointing & Storage
SAVE_TO_DRIVE = True  # @param {type:"boolean"}
SAVE_EVERY_N_STEPS = 200  # @param {type:"integer"}
MAX_CHECKPOINTS_TO_KEEP = 3  # @param {type:"integer"}

# @markdown ### 🚀 Training Controls
BATCH_SIZE = 8  # @param [4, 8, 16, 32]
EPOCHS = 3  # @param {type:"integer"}
LEARNING_RATE = 3e-4  # @param {type:"number"}
ENABLE_MIXED_PRECISION = True  # @param {type:"boolean"}

# @markdown ### 🌐 Hugging Face Hub (Optional Publishing)
PUSH_TO_HUB = False  # @param {type:"boolean"}
HF_TOKEN = ""  # @param {type:"string"}
HF_REPO_ID = "my-babyflash-model"  # @param {type:"string"}

# 1. Silent Automated Dependency Installation
print("Installing core dependencies (silently)...")
import subprocess
import sys
import os
deps = ["transformers", "datasets", "accelerate", "sentencepiece", "pillow", "einops"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + deps)
print("✓ Dependencies verified.")

# 2. Automated Google Drive Mount Engine
BASE_CHECKPOINT_DIR = "./BabyAI_Checkpoints"
if SAVE_TO_DRIVE:
    try:
        # Detect if running in Google Colab environment
        if 'google.colab' in sys.modules or os.path.exists('/content'):
            from google.colab import drive
            drive.mount('/content/drive')
            BASE_CHECKPOINT_DIR = "/content/drive/MyDrive/BabyAI_Checkpoints"
            print("✓ Google Drive mounted successfully at /content/drive")
        else:
            print("Notice: Running locally. Checkpoints will be saved to ./BabyAI_Checkpoints")
    except Exception as e:
        print(f"Notice: Google Drive mount skipped ({e}). Checkpoints will save to ./BabyAI_Checkpoints")

os.makedirs(BASE_CHECKPOINT_DIR, exist_ok=True)
print(f"✓ Base Checkpoint Directory: {BASE_CHECKPOINT_DIR}")

# 3. Hardware Audit
import torch
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Active Compute Device: {device.upper()}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Hardware: {torch.cuda.get_device_name(0)} ({vram_gb:.2f} GB VRAM)")
    torch.cuda.empty_cache()
else:
    print("Mode: CPU (Low-memory guardrails and gradient checkpoints active)")
print("=" * 60)


In [ ]:
# @title 📊 Cell 2: Universal Data Ingestion & Directory Sanitization
import os
import re
from typing import Dict, Any
from datasets import load_dataset
from transformers import AutoTokenizer

# 1. Directory Sanitization & Unique Run Identification
def sanitize_identifier(name: str) -> str:
    cleaned = re.sub(r'[^a-zA-Z0-9_]', '_', str(name).strip())
    cleaned = re.sub(r'_+', '_', cleaned).strip('_')
    return cleaned or "unnamed"

UNIQUE_RUN_ID = f"{sanitize_identifier(MODEL_NAME)}_on_{sanitize_identifier(DATASET_PATH)}"
RUN_CHECKPOINT_DIR = os.path.join(BASE_CHECKPOINT_DIR, UNIQUE_RUN_ID)
os.makedirs(RUN_CHECKPOINT_DIR, exist_ok=True)

print("=" * 60)
print(f"✓ Unique Training Run ID: '{UNIQUE_RUN_ID}'")
print(f"✓ Target Checkpoint Folder: '{RUN_CHECKPOINT_DIR}'")
print("=" * 60)

# 2. Tokenizer Setup with Special Multi-Modal & Agent Tokens
TOKENIZER_NAME = "Qwen/Qwen2.5-0.5B"
print(f"Loading tokenizer: {TOKENIZER_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)

SPECIAL_TOKENS = {
    "additional_special_tokens": [
        "<think>", "</think>",                      # DeepSeek-R1 / V4 Reasoning tokens
        "<tool_call>", "</tool_call>",              # Tool call tokens
        "<tool_response>", "</tool_response>",      # Tool execution response tokens
        "<image>", "</image>",                      # Vision patch anchor tokens
    ]
}
num_added = tokenizer.add_special_tokens(SPECIAL_TOKENS)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

TOTAL_VOCAB_SIZE = max(len(tokenizer), tokenizer.vocab_size)
print(f"✓ Tokenizer ready with {num_added} special tokens | Total Vocab Size: {TOTAL_VOCAB_SIZE:,}")

# 3. Universal Dual-Input Data Pipeline
print(f"\nIngesting dataset via Mode: '{DATASET_SOURCE}' from '{DATASET_PATH}' for task '{TASK_TYPE}'...")

def format_sample(row: Dict[str, Any], task: str) -> str:
    cols = row.keys()
    inst = next((c for c in ["instruction", "prompt", "question", "problem"] if c in cols), None)
    out = next((c for c in ["output", "response", "answer", "solution"] if c in cols), None)
    inp = next((c for c in ["input", "context"] if c in cols), None)

    if task == "reasoning":
        q = str(row[inst] or "Analyze this problem.") if inst else "Analyze this problem."
        ans = str(row[out] or row.get("text") or "")
        return f"### Problem:\n{q}\n\n### Solution:\n<think>Analyzing constraints and intermediate steps.</think>\n{ans}"

    elif task in ["coding", "tool_use"]:
        req = str(row[inst] or "Perform operation.") if inst else "Perform operation."
        ans = str(row[out] or row.get("text") or "")
        tool_query = req[:40]
        tool_json = f'{{"action": "execute", "query": "{tool_query}"}}'
        return f"### Request:\n{req}\n\n### Agent Action:\n<tool_call>{tool_json}</tool_call>\n<tool_response>Success</tool_response>\n{ans}"

    elif task == "vision":
        cap = str(row.get("text") or row.get("caption") or row.get("response") or "A descriptive visual scene.")
        return f"<image> Describe image: {cap}"

    # Default Instruction / Text
    if inst and out:
        c = f"\n\n### Context:\n{row[inp]}" if inp and row[inp] else ""
        return f"### Instruction:\n{row[inst]}{c}\n\n### Response:\n{row[out]}"

    text_col = next((c for c in ["text", "content", "body", "article"] if c in cols), None)
    if text_col:
        return str(row[text_col] or "").strip()

    return " ".join(str(v) for v in row.values() if v is not None)

# Ingest data from HF Hub or Local / Drive file
if DATASET_SOURCE == "Custom Upload / Drive" or os.path.exists(DATASET_PATH):
    ext = os.path.splitext(DATASET_PATH)[-1].lower()
    type_map = {".json": "json", ".jsonl": "json", ".csv": "csv", ".tsv": "csv", ".parquet": "parquet", ".txt": "text"}
    raw_dataset = load_dataset(type_map.get(ext, "text"), data_files=DATASET_PATH, split="train")
else:
    raw_dataset = load_dataset(DATASET_PATH, split="train")

if MAX_SAMPLES is not None and len(raw_dataset) > MAX_SAMPLES:
    raw_dataset = raw_dataset.shuffle(seed=42).select(range(MAX_SAMPLES))

print(f"✓ Ingested {len(raw_dataset):,} samples.")
print("\n--- Sample Formatted Input Preview ---")
print(format_sample(raw_dataset[0], TASK_TYPE)[:250] + "...\n" + "-" * 38)

def tokenize_batch(examples):
    keys = list(examples.keys())
    batch_len = len(examples[keys[0]])
    formatted = []
    for i in range(batch_len):
        row = {k: examples[k][i] for k in keys}
        formatted.append(format_sample(row, TASK_TYPE) + tokenizer.eos_token)

    tokens = tokenizer(
        formatted,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        return_tensors=None,
    )
    return {
        "input_ids": tokens["input_ids"],
        "attention_mask": tokens["attention_mask"],
    }

tokenized_dataset = raw_dataset.map(
    tokenize_batch,
    batched=True,
    batch_size=1000,
    remove_columns=raw_dataset.column_names,
    desc="Tokenizing for BabyFlash Causal LM",
)
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask"])
print(f"✓ Pipeline tokenized {len(tokenized_dataset):,} samples. Tensor shape: {tokenized_dataset[0]['input_ids'].shape}")


In [ ]:
# @title 🧠 Cell 3: Fully-Implemented PyTorch DeepSeek Engine
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict, Union
from transformers.configuration_utils import PretrainedConfig
from transformers.modeling_utils import PreTrainedModel
from transformers.modeling_outputs import CausalLMOutputWithPast

# 1. Architecture Configuration
SCALE_CONFIGS = {
    "Baby-Tiny": {
        "hidden_size": 192,
        "num_encoder_layers": 2,
        "num_decoder_layers": 4,
        "num_attention_heads": 6,
        "kv_compression_dim": 48,
        "rope_dim": 16,
        "intermediate_size": 384,
        "encoder_latent_dim": 96,
        "num_routed_experts": 8,
        "num_experts_per_tok": 2,
        "num_shared_experts": 1,
    },
    "Baby-Flash": {
        "hidden_size": 256,
        "num_encoder_layers": 4,
        "num_decoder_layers": 6,
        "num_attention_heads": 8,
        "kv_compression_dim": 64,
        "rope_dim": 16,
        "intermediate_size": 512,
        "encoder_latent_dim": 128,
        "num_routed_experts": 8,
        "num_experts_per_tok": 2,
        "num_shared_experts": 1,
    }
}
ACTIVE_SCALE = SCALE_CONFIGS.get(MODEL_NAME, SCALE_CONFIGS["Baby-Flash"])

class BabyFlashConfig(PretrainedConfig):
    model_type = "baby_flash"
    def __init__(
        self,
        vocab_size: int = 151665,
        hidden_size: int = ACTIVE_SCALE["hidden_size"],
        num_encoder_layers: int = ACTIVE_SCALE["num_encoder_layers"],
        num_decoder_layers: int = ACTIVE_SCALE["num_decoder_layers"],
        num_attention_heads: int = ACTIVE_SCALE["num_attention_heads"],
        kv_compression_dim: int = ACTIVE_SCALE["kv_compression_dim"],
        rope_dim: int = ACTIVE_SCALE["rope_dim"],
        intermediate_size: int = ACTIVE_SCALE["intermediate_size"],
        encoder_latent_dim: int = ACTIVE_SCALE["encoder_latent_dim"],
        num_routed_experts: int = ACTIVE_SCALE["num_routed_experts"],
        num_experts_per_tok: int = ACTIVE_SCALE["num_experts_per_tok"],
        num_shared_experts: int = ACTIVE_SCALE["num_shared_experts"],
        router_aux_loss_coef: float = 0.01,
        image_size: int = 224,
        patch_size: int = 16,
        max_position_embeddings: int = 2048,
        initializer_range: float = 0.02,
        rms_norm_eps: float = 1e-6,
        use_cache: bool = True,
        pad_token_id: int = 151643,
        eos_token_id: int = 151643,
        tie_word_embeddings: bool = True,
        **kwargs,
    ):
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_encoder_layers = num_encoder_layers
        self.num_decoder_layers = num_decoder_layers
        self.num_attention_heads = num_attention_heads
        self.kv_compression_dim = kv_compression_dim
        self.rope_dim = rope_dim
        self.intermediate_size = intermediate_size
        self.encoder_latent_dim = encoder_latent_dim
        self.num_routed_experts = num_routed_experts
        self.num_experts_per_tok = num_experts_per_tok
        self.num_shared_experts = num_shared_experts
        self.router_aux_loss_coef = router_aux_loss_coef
        self.image_size = image_size
        self.patch_size = patch_size
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.rms_norm_eps = rms_norm_eps
        self.use_cache = use_cache
        super().__init__(pad_token_id=pad_token_id, eos_token_id=eos_token_id, tie_word_embeddings=tie_word_embeddings, **kwargs)

# 2. Normalization, RoPE & Vision Patch Projection
class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = x.pow(2).mean(dim=-1, keepdim=True)
        return self.weight * x * torch.rsqrt(rms + self.eps)

def rotate_half(x: torch.Tensor) -> torch.Tensor:
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_emb(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    return (x * cos) + (rotate_half(x) * sin)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim: int, max_position_embeddings: int = 2048, base: float = 10000.0):
        super().__init__()
        self.dim = dim
        self.max_position_embeddings = max_position_embeddings
        self.base = base
        inv_freq = 1.0 / (self.base ** (torch.arange(0, self.dim, 2).float() / self.dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, seq_len: int, device: torch.device, offset: int = 0) -> Tuple[torch.Tensor, torch.Tensor]:
        t = torch.arange(offset, offset + seq_len, device=device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        cos = emb.cos()[None, None, :, :]
        sin = emb.sin()[None, None, :, :]
        return cos, sin

class VisionPatchEmbedding(nn.Module):
    def __init__(self, config: BabyFlashConfig):
        super().__init__()
        self.image_size = config.image_size
        self.patch_size = config.patch_size
        self.proj = nn.Conv2d(3, config.hidden_size, kernel_size=self.patch_size, stride=self.patch_size, bias=False)
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        x = self.proj(pixel_values).flatten(2).transpose(1, 2)
        return self.norm(x)

# 3. Compressed Attention (Multi-Head Latent Attention)
class CompressedAttention(nn.Module):
    def __init__(self, config: BabyFlashConfig, layer_idx: int = 0):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.kv_comp_dim = config.kv_compression_dim
        self.rope_dim = config.rope_dim

        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.q_rope_proj = nn.Linear(self.hidden_size, self.num_heads * self.rope_dim, bias=False)
        self.kv_down_proj = nn.Linear(self.hidden_size, self.kv_comp_dim, bias=False)
        self.kv_norm = RMSNorm(self.kv_comp_dim, eps=config.rms_norm_eps)
        self.k_up_proj = nn.Linear(self.kv_comp_dim, self.num_heads * self.head_dim, bias=False)
        self.v_up_proj = nn.Linear(self.kv_comp_dim, self.num_heads * self.head_dim, bias=False)
        self.k_rope_proj = nn.Linear(self.hidden_size, self.rope_dim, bias=False)
        self.out_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)
        self.rotary = RotaryEmbedding(self.rope_dim, max_position_embeddings=config.max_position_embeddings)

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        past_key_value: Optional[Dict[str, torch.Tensor]] = None,
        use_cache: bool = False,
    ) -> Tuple[torch.Tensor, Optional[Dict[str, torch.Tensor]]]:
        B, T, _ = hidden_states.shape
        q_c = self.q_proj(hidden_states).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        q_r = self.q_rope_proj(hidden_states).view(B, T, self.num_heads, self.rope_dim).transpose(1, 2)

        c_kv = self.kv_norm(self.kv_down_proj(hidden_states))
        k_r = self.k_rope_proj(hidden_states).view(B, T, 1, self.rope_dim).transpose(1, 2)

        offset = 0
        if past_key_value is not None and "c_kv" in past_key_value:
            offset = past_key_value["c_kv"].shape[1]
            c_kv = torch.cat([past_key_value["c_kv"], c_kv], dim=1)
            k_r = torch.cat([past_key_value["k_rope"], k_r], dim=2)

        present_key_value = {"c_kv": c_kv, "k_rope": k_r} if use_cache else None
        total_len = c_kv.shape[1]

        cos, sin = self.rotary(total_len, device=hidden_states.device)
        q_r = apply_rotary_emb(q_r, cos[:, :, offset : offset + T, :], sin[:, :, offset : offset + T, :])
        k_r = apply_rotary_emb(k_r, cos, sin)

        k_c = self.k_up_proj(c_kv).view(B, total_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_up_proj(c_kv).view(B, total_len, self.num_heads, self.head_dim).transpose(1, 2)

        k_r = k_r.repeat(1, self.num_heads, 1, 1)
        q = torch.cat([q_c, q_r], dim=-1)
        k = torch.cat([k_c, k_r], dim=-1)

        is_causal = (T > 1 and past_key_value is None and attention_mask is None)
        attn_mask = None
        if attention_mask is not None and attention_mask.dim() == 2:
            attn_mask = attention_mask[:, None, None, :].to(dtype=q.dtype)
            attn_mask = (1.0 - attn_mask) * -10000.0

        attn_out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, is_causal=is_causal)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, self.num_heads * self.head_dim)
        return self.out_proj(attn_out), present_key_value

# 4. SwiGLU & Sparse MoE Layer
class SwiGLU(nn.Module):
    def __init__(self, dim: int, hidden_dim: int):
        super().__init__()
        self.gate = nn.Linear(dim, hidden_dim, bias=False)
        self.up = nn.Linear(dim, hidden_dim, bias=False)
        self.down = nn.Linear(hidden_dim, dim, bias=False)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down(F.silu(self.gate(x)) * self.up(x))

class MoELayer(nn.Module):
    def __init__(self, config: BabyFlashConfig):
        super().__init__()
        self.config = config
        self.router = nn.Linear(config.hidden_size, config.num_routed_experts, bias=False)
        self.routed_experts = nn.ModuleList([
            SwiGLU(config.hidden_size, config.intermediate_size)
            for _ in range(config.num_routed_experts)
        ])
        self.shared_experts = nn.ModuleList([
            SwiGLU(config.hidden_size, config.intermediate_size)
            for _ in range(config.num_shared_experts)
        ])
        self.top_k = config.num_experts_per_tok
        self.aux_coef = config.router_aux_loss_coef

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, C = x.shape
        x_flat = x.view(-1, C)
        probs = F.softmax(self.router(x_flat), dim=-1)
        top_w, top_ids = torch.topk(probs, self.top_k, dim=-1)
        top_w = top_w / top_w.sum(dim=-1, keepdim=True)

        if self.training and x_flat.shape[0] > 0:
            P_avg = probs.mean(dim=0)
            one_hot = F.one_hot(top_ids, num_classes=self.config.num_routed_experts).float()
            f_avg = one_hot.sum(dim=1).mean(dim=0) / self.top_k
            aux_loss = self.aux_coef * self.config.num_routed_experts * (P_avg * f_avg).sum()
        else:
            aux_loss = torch.tensor(0.0, device=x.device, dtype=x.dtype)

        routed_out = torch.zeros_like(x_flat)
        for exp_id, exp in enumerate(self.routed_experts):
            mask = (top_ids == exp_id)
            if not mask.any():
                continue
            t_idx, c_idx = mask.nonzero(as_tuple=True)
            sel = x_flat[t_idx]
            w = top_w[t_idx, c_idx].unsqueeze(-1)
            routed_out = routed_out.index_put((t_idx,), exp(sel) * w, accumulate=True)

        shared_out = torch.zeros_like(x_flat)
        for shared_exp in self.shared_experts:
            shared_out = shared_out + shared_exp(x_flat)

        return (routed_out + shared_out).view(B, T, C), aux_loss

# 5. Causal Encoder & Decoder Blocks
class CausalEncoderBlock(nn.Module):
    def __init__(self, config: BabyFlashConfig, layer_idx: int = 0):
        super().__init__()
        self.norm1 = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.attn = CompressedAttention(config, layer_idx=layer_idx)
        self.norm2 = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.mlp = SwiGLU(config.hidden_size, config.intermediate_size)
    def forward(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        x = x + self.attn(self.norm1(x), attention_mask=attention_mask, use_cache=False)[0]
        x = x + self.mlp(self.norm2(x))
        return x

class CausalDecoderBlock(nn.Module):
    def __init__(self, config: BabyFlashConfig, layer_idx: int = 0):
        super().__init__()
        self.norm1 = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.attn = CompressedAttention(config, layer_idx=layer_idx)
        self.norm2 = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.moe = MoELayer(config)
    def forward(self, x: torch.Tensor, attention_mask: Optional[torch.Tensor] = None, past_key_value: Optional[Dict[str, torch.Tensor]] = None, use_cache: bool = False):
        attn_out, present_kv = self.attn(self.norm1(x), attention_mask=attention_mask, past_key_value=past_key_value, use_cache=use_cache)
        x = x + attn_out
        moe_out, aux_loss = self.moe(self.norm2(x))
        x = x + moe_out
        return x, aux_loss, present_kv

# 6. Full Multi-Modal Model & Causal LM Wrapper
@dataclass
class BabyFlashOutput(CausalLMOutputWithPast):
    router_aux_loss: Optional[torch.FloatTensor] = None

class BabyFlashMultiModalModel(nn.Module):
    def __init__(self, config: BabyFlashConfig):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size, padding_idx=config.pad_token_id)
        self.embed_norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.vision_proj = VisionPatchEmbedding(config)

        self.encoder = nn.ModuleList([CausalEncoderBlock(config, i) for i in range(config.num_encoder_layers)])
        self.enc_to_dec = nn.Linear(config.hidden_size, config.encoder_latent_dim, bias=False)
        self.dec_latent = nn.Linear(config.encoder_latent_dim, config.hidden_size, bias=False)
        self.decoder = nn.ModuleList([CausalDecoderBlock(config, i) for i in range(config.num_decoder_layers)])
        self.final_norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, input_ids: Optional[torch.LongTensor] = None, pixel_values: Optional[torch.FloatTensor] = None, attention_mask: Optional[torch.Tensor] = None, past_key_values: Optional[List[Dict[str, torch.Tensor]]] = None, use_cache: bool = True):
        h = self.embed_norm(self.embed_tokens(input_ids)) if input_ids is not None else None
        if pixel_values is not None:
            vis = self.vision_proj(pixel_values)
            if h is not None:
                h = torch.cat([vis, h], dim=1)
                if attention_mask is not None:
                    vm = torch.ones((pixel_values.shape[0], vis.shape[1]), device=pixel_values.device)
                    attention_mask = torch.cat([vm, attention_mask], dim=1)
            else:
                h = vis

        total_aux = torch.tensor(0.0, device=h.device, dtype=h.dtype)
        present_kvs = [] if use_cache else None

        if past_key_values is None or len(past_key_values) == 0:
            enc_h = h
            for enc_layer in self.encoder:
                enc_h = enc_layer(enc_h, attention_mask=attention_mask)
            h = h + self.dec_latent(self.enc_to_dec(enc_h))

        for idx, dec_layer in enumerate(self.decoder):
            p_kv = past_key_values[idx] if past_key_values is not None else None
            h, aux_loss, cur_kv = dec_layer(h, attention_mask=attention_mask, past_key_value=p_kv, use_cache=use_cache)
            total_aux = total_aux + aux_loss
            if use_cache:
                present_kvs.append(cur_kv)

        return self.final_norm(h), total_aux, present_kvs

class BabyFlashMultiModalForCausalLM(PreTrainedModel):
    config_class = BabyFlashConfig
    base_model_prefix = "model"
    _tied_weights_keys = ["lm_head.weight"]

    def __init__(self, config: BabyFlashConfig):
        super().__init__(config)
        self.model = BabyFlashMultiModalModel(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        if config.tie_word_embeddings:
            self.lm_head.weight = self.model.embed_tokens.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.Embedding):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)

    def forward(self, input_ids: Optional[torch.LongTensor] = None, pixel_values: Optional[torch.FloatTensor] = None, attention_mask: Optional[torch.Tensor] = None, past_key_values: Optional[List[Dict[str, torch.Tensor]]] = None, labels: Optional[torch.LongTensor] = None, use_cache: bool = True, return_dict: bool = True, **kwargs):
        h, aux_loss, present_kvs = self.model(input_ids, pixel_values, attention_mask, past_key_values, use_cache)
        logits = self.lm_head(h)

        loss = None
        if labels is not None:
            s_logits = logits[..., :-1, :].contiguous()
            s_labels = labels[..., 1:].contiguous()
            if s_labels.shape[1] < s_logits.shape[1]:
                s_logits = s_logits[:, s_logits.shape[1] - s_labels.shape[1]:, :]
            lm_loss = F.cross_entropy(s_logits.view(-1, self.config.vocab_size), s_labels.view(-1), ignore_index=-100)
            loss = lm_loss + aux_loss

        return BabyFlashOutput(loss=loss, logits=logits, past_key_values=present_kvs, router_aux_loss=aux_loss)

    def prepare_inputs_for_generation(self, input_ids, past_key_values=None, attention_mask=None, **kwargs):
        if past_key_values is not None:
            input_ids = input_ids[:, -1:]
        return {"input_ids": input_ids, "past_key_values": past_key_values, "attention_mask": attention_mask, "use_cache": True}

    @torch.no_grad()
    def generate(self, input_ids: torch.LongTensor, pixel_values: Optional[torch.FloatTensor] = None, max_new_tokens: int = 60, temperature: float = 0.7, top_k: int = 50, top_p: float = 0.9, eos_token_id: Optional[int] = None) -> torch.LongTensor:
        self.eval()
        eos_id = eos_token_id or self.config.eos_token_id
        gen = input_ids.clone()
        p_kvs = None
        curr_ids = input_ids
        first = True

        for _ in range(max_new_tokens):
            if first and pixel_values is not None:
                out = self(input_ids=curr_ids, pixel_values=pixel_values, past_key_values=p_kvs, use_cache=True)
                first = False
            else:
                out = self(input_ids=curr_ids, past_key_values=p_kvs, use_cache=True)

            logits = out.logits[:, -1, :]
            p_kvs = out.past_key_values

            if temperature > 0:
                logits = logits / temperature
                if top_k > 0:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, [-1]]] = -float("Inf")
                if 0.0 < top_p < 1.0:
                    s_logits, s_indices = torch.sort(logits, descending=True)
                    cum_probs = torch.cumsum(F.softmax(s_logits, dim=-1), dim=-1)
                    s_mask = cum_probs > top_p
                    s_mask[..., 1:] = s_mask[..., :-1].clone()
                    s_mask[..., 0] = 0
                    mask = s_mask.scatter(1, s_indices, s_mask)
                    logits[mask] = -float("Inf")
                probs = F.softmax(logits, dim=-1)
                next_tok = torch.multinomial(probs, num_samples=1)
            else:
                next_tok = torch.argmax(logits, dim=-1, keepdim=True)

            gen = torch.cat([gen, next_tok], dim=1)
            curr_ids = next_tok
            if eos_id is not None and (next_tok == eos_id).all():
                break

        return gen

# Instantiate Model
config = BabyFlashConfig(
    vocab_size=TOTAL_VOCAB_SIZE,
    pad_token_id=tokenizer.pad_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
model = BabyFlashMultiModalForCausalLM(config).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 60)
print(f"✓ DeepSeek V4.1 Flash Engine Initialized ({MODEL_NAME}) on: {device.upper()}")
print(f"Total Parameters:       {total_params:,} ({total_params/1e6:.2f}M)")
print(f"Trainable Parameters:   {trainable_params:,}")
print(f"Causal Encoder/Decoder: {config.num_encoder_layers}L prefill / {config.num_decoder_layers}L generation")
print(f"MoE Activation:         {config.num_routed_experts} Routed (Top-{config.num_experts_per_tok}) + {config.num_shared_experts} Shared Expert")
print(f"KV Cache Compression:   Rank {config.kv_compression_dim} (MLA memory efficiency: ~890B/tok)")
print("=" * 60)


In [ ]:
# @title 🚀 Cell 4: Smart Training Engine with Auto-Resume
import gc
import os
import shutil
import glob
from torch.utils.data import DataLoader

# 1. Setup Optimizer and Mixed-Precision Scaler
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
use_amp = ENABLE_MIXED_PRECISION and (device == "cuda")
if device == "cuda":
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
else:
    scaler = torch.amp.GradScaler("cpu", enabled=False)

train_loader = DataLoader(tokenized_dataset, batch_size=BATCH_SIZE, shuffle=True)
steps_per_epoch = len(train_loader)
total_target_steps = EPOCHS * steps_per_epoch

# 2. Inspect Existing Checkpoints & Auto-Resume Logic
existing_ckpts = []
if os.path.exists(RUN_CHECKPOINT_DIR):
    for d in os.listdir(RUN_CHECKPOINT_DIR):
        if d.startswith("checkpoint-step-"):
            try:
                s_num = int(d.split("-")[-1])
                existing_ckpts.append((s_num, os.path.join(RUN_CHECKPOINT_DIR, d)))
            except ValueError:
                pass
    existing_ckpts.sort(key=lambda x: x[0])

start_step = 0
start_epoch = 0
loss_history = []

if existing_ckpts:
    latest_step, latest_ckpt_dir = existing_ckpts[-1]
    print("=" * 60)
    print(f"[✓] Existing training session found for Model '{MODEL_NAME}' on Dataset '{DATASET_PATH}'.")
    print(f"[✓] Resuming seamlessly from checkpoint: {latest_ckpt_dir} (Step {latest_step:,})...")
    print("=" * 60)

    try:
        # Load Model Weights
        weight_file = os.path.join(latest_ckpt_dir, "pytorch_model.bin")
        if os.path.exists(weight_file):
            model.load_state_dict(torch.load(weight_file, map_location=device))
        else:
            model = BabyFlashMultiModalForCausalLM.from_pretrained(latest_ckpt_dir).to(device)

        # Load Optimizer, Scaler, and Training State
        state_file = os.path.join(latest_ckpt_dir, "training_state.pt")
        if os.path.exists(state_file):
            state = torch.load(state_file, map_location=device)
            optimizer.load_state_dict(state["optimizer_state_dict"])
            if use_amp and state.get("scaler_state_dict") is not None:
                scaler.load_state_dict(state["scaler_state_dict"])
            start_step = state.get("step", latest_step)
            start_epoch = state.get("epoch", start_step // steps_per_epoch)
            loss_history = state.get("loss_history", [])

        print(f"✓ Resumed successfully! Continuing from Epoch {start_epoch + 1}, Global Step {start_step:,}...")
    except Exception as e:
        print(f"Notice: Auto-resume hit an error ({e}). Starting fresh from step 0.")
        start_step = 0
        start_epoch = 0
else:
    print("=" * 60)
    print(f"[+] New unique combination detected: '{UNIQUE_RUN_ID}'.")
    print("[+] Starting fresh training run with zero previous checkpoints...")
    print("=" * 60)

# 3. Rolling Checkpoint Saver Function (save_total_limit = 3)
def save_checkpoint(curr_step: int, curr_epoch: int, is_emergency: bool = False):
    tag = f"checkpoint-step-{curr_step}" if not is_emergency else f"checkpoint-emergency-step-{curr_step}"
    save_path = os.path.join(RUN_CHECKPOINT_DIR, tag)
    os.makedirs(save_path, exist_ok=True)

    # Save Model Weights & Tokenizer Config (safe_serialization=False preserves tied weights)
    model._tied_weights_keys = ["lm_head.weight"]
    try:
        model.save_pretrained(save_path, safe_serialization=False)
    except Exception:
        torch.save(model.state_dict(), os.path.join(save_path, "pytorch_model.bin"))
        config.save_pretrained(save_path)

    tokenizer.save_pretrained(save_path)

    # Save Training State (Optimizer, Scaler, Step, Epoch, Loss)
    state = {
        "step": curr_step,
        "epoch": curr_epoch,
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict() if use_amp else None,
        "loss_history": loss_history,
        "unique_run_id": UNIQUE_RUN_ID,
    }
    torch.save(state, os.path.join(save_path, "training_state.pt"))
    status_label = "EMERGENCY" if is_emergency else "SCHEDULED"
    print(f"\n💾 [{status_label} SAVE] Checkpoint synced to Drive: {save_path}")

    # Enforce Rolling Limit to Conserve Google Drive Storage
    all_ckpts = []
    for d in os.listdir(RUN_CHECKPOINT_DIR):
        if d.startswith("checkpoint-step-"):
            try:
                s_num = int(d.split("-")[-1])
                all_ckpts.append((s_num, os.path.join(RUN_CHECKPOINT_DIR, d)))
            except ValueError:
                pass
    all_ckpts.sort(key=lambda x: x[0])

    while len(all_ckpts) > MAX_CHECKPOINTS_TO_KEEP:
        oldest_step, oldest_dir = all_ckpts.pop(0)
        shutil.rmtree(oldest_dir, ignore_errors=True)
        print(f"🧹 [STORAGE OPTIMIZER] Pruned older checkpoint to save Drive space: {oldest_dir}")

# 4. Training Loop with OOM Guardrails and Graceful Interruption Handler
model.train()
global_step = start_step

try:
    for epoch in range(start_epoch, EPOCHS):
        epoch_loss, epoch_aux = 0.0, 0.0
        successful_steps = 0

        for batch_idx, batch in enumerate(train_loader):
            # If resuming mid-epoch, skip already completed batches
            current_batch_global_step = epoch * steps_per_epoch + batch_idx
            if current_batch_global_step < start_step:
                continue

            try:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)

                labels = input_ids.clone()
                if tokenizer.pad_token_id is not None:
                    labels[labels == tokenizer.pad_token_id] = -100

                pixel_values = None
                if TASK_TYPE == "vision" and (batch_idx % 2 == 0):
                    pixel_values = torch.zeros((input_ids.shape[0], 3, config.image_size, config.image_size), device=device)

                optimizer.zero_grad()

                autocast_device = "cuda" if device == "cuda" else "cpu"
                with torch.amp.autocast(autocast_device, enabled=use_amp):
                    outputs = model(
                        input_ids=input_ids,
                        pixel_values=pixel_values,
                        attention_mask=attention_mask,
                        labels=labels,
                    )
                    loss = outputs.loss

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

                epoch_loss += loss.item()
                aux_val = outputs.router_aux_loss.item() if outputs.router_aux_loss is not None else 0.0
                epoch_aux += aux_val
                successful_steps += 1
                global_step += 1
                loss_history.append((global_step, round(loss.item(), 4)))

                if global_step % 50 == 0 or batch_idx == 0:
                    print(f"Epoch [{epoch+1}/{EPOCHS}] Step [{global_step:>5d}/{total_target_steps}] Loss: {loss.item():.4f} (MoE Aux: {aux_val:.4f})")

                # Periodic Checkpoint Saving to Google Drive
                if global_step % SAVE_EVERY_N_STEPS == 0:
                    save_checkpoint(global_step, epoch, is_emergency=False)

            except torch.cuda.OutOfMemoryError:
                print(f"⚠️ OOM intercepted at step {global_step}. Purging CUDA cache and resuming...")
                gc.collect()
                torch.cuda.empty_cache()
                optimizer.zero_grad()
                continue

        avg_loss = epoch_loss / max(1, successful_steps)
        avg_aux = epoch_aux / max(1, successful_steps)
        print(f"\n>>> Epoch {epoch+1} Complete | Average Loss = {avg_loss:.4f} | MoE Aux = {avg_aux:.4f}\n")

    # Final Checkpoint Save
    save_checkpoint(global_step, EPOCHS, is_emergency=False)
    print(f"🎉 Training fully completed! Final model synced to: {RUN_CHECKPOINT_DIR}")

except KeyboardInterrupt:
    print("\n" + "!" * 60)
    print("⚠️ Training paused by user! Triggering emergency state save...")
    print("!" * 60)
    save_checkpoint(global_step, epoch, is_emergency=True)
    print("✓ Emergency checkpoint saved cleanly to Google Drive.")
    print(f"✓ You can close your browser or disconnect. Re-running this cell will resume from step {global_step:,}!")


In [ ]:
# @title 💬 Cell 5: Test, Infer & Publish Playground
# @markdown Test generation across modalities and optionally publish to Hugging Face Hub.

TEST_PROMPT = "Once upon a time"  # @param {type:"string"}
MAX_NEW_TOKENS = 60  # @param {type:"integer"}
TEMPERATURE = 0.7  # @param {type:"number"}
TOP_K = 50  # @param {type:"integer"}

def test_inference(prompt: str, is_vision: bool = False):
    model.eval()
    print(f"\n[Test Prompt]: {prompt}")
    input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)

    pixel_values = None
    if is_vision:
        pixel_values = torch.randn(1, 3, config.image_size, config.image_size, device=device)

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            pixel_values=pixel_values,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_k=TOP_K,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(output_ids[0], skip_special_tokens=False)
    print(f"\n[BabyFlash Output]:\n{response}\n")
    return response

# 1. Main prompt test
test_inference(TEST_PROMPT, is_vision=(TASK_TYPE == "vision"))

# 2. Specialized Task Verification
if TASK_TYPE == "reasoning":
    print("--- Structured Reasoning Verification (<think> tag) ---")
    test_inference("### Problem:\nHow many edges does a cube have?\n### Solution:\n")

elif TASK_TYPE in ["coding", "tool_use"]:
    print("--- Agent Tool Execution Verification (<tool_call> tag) ---")
    test_inference("### Request:\nCheck the weather in Tokyo.\n### Agent Action:\n")

# 3. 1-Click Hugging Face Hub Publishing
if PUSH_TO_HUB and HF_TOKEN and HF_REPO_ID:
    try:
        from huggingface_hub import login
        print(f"\nAuthenticating with Hugging Face Hub...")
        login(token=HF_TOKEN)
        print(f"Pushing model & tokenizer to repo: '{HF_REPO_ID}'...")
        model.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
        tokenizer.push_to_hub(HF_REPO_ID, token=HF_TOKEN)
        print(f"🎉 Model published! Check it out at: https://huggingface.co/{HF_REPO_ID}")
    except Exception as e:
        print(f"Notice: Hub publish encountered: {e}")
elif PUSH_TO_HUB:
    print("\nNotice: Check PUSH_TO_HUB and provide HF_TOKEN and HF_REPO_ID in Cell 1 to publish.")
